# Fase 1 Formativa: definición del proyecto y plan inicial de EDA reproducible

**Asignatura:** MCDI503 · Exploración Inteligente para la Ciencia de Datos  
**Integrantes:** Gonzalo Bouldres · Eduardo Contreras · Luis Díaz  
**Dataset:** `titanic` distribuido mediante Seaborn

## I. Título tentativo del proyecto

**Análisis exploratorio de variables asociadas a la supervivencia de pasajeros del Titanic**

El notebook documenta la caracterización inicial del conjunto de datos y la planificación del análisis exploratorio de datos (EDA). El alcance de esta fase comprende la revisión técnica del insumo, la formulación de preguntas exploratorias, la definición de supuestos y limitaciones preliminares, la organización prevista del análisis y los criterios de reproducibilidad.


## II. Descripción del proyecto

El proyecto estudia el conjunto de datos `titanic`, que reúne información demográfica y del viaje de pasajeros del Titanic junto con su condición de supervivencia. El propósito es establecer una base analítica reproducible para estudiar descriptivamente variables como `sex`, `pclass`, `age`, `fare` y características familiares.

El análisis busca apoyar la comprensión estructurada de patrones presentes en los datos mediante una secuencia de trabajo documentada, reproducible y limitada a asociaciones descriptivas, sin atribución causal.


## III. Descripción del conjunto de datos

### 3.1 Carga controlada e integridad

Se utiliza una copia local del dataset como insumo de análisis. El archivo de entrada se conserva sin sobrescritura y se identifica mediante un hash SHA-256, con el propósito de mantener trazabilidad sobre la versión exacta de los datos utilizada.


In [1]:
from pathlib import Path
import hashlib
import platform
import sys

import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name.lower() in {"notebook", "notebooks"}:
    ROOT = ROOT.parent

INPUT_FILE = ROOT / "data" / "input" / "titanic_seaborn.csv"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA256 = "43af81a01f25a7771c9a42809a31625f4d0de3f335cdf29961f9ba179a86c13d"

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        "No se encontró data/input/titanic_seaborn.csv. "
        "La Formativa utiliza una copia local controlada y no descarga datos durante la ejecución."
    )

sha256 = hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest()
if sha256 != EXPECTED_SHA256:
    raise ValueError("El SHA-256 del dataset no coincide con la copia auditada del proyecto.")

df = pd.read_csv(INPUT_FILE)

print("Raíz lógica del proyecto: ./")
print("Archivo de entrada: data/input/titanic_seaborn.csv")
print("SHA-256:", sha256)
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("seaborn:", sns.__version__)
print("Sistema:", platform.system())
print("Dimensiones:", df.shape)


Raíz lógica del proyecto: ./
Archivo de entrada: data/input/titanic_seaborn.csv
SHA-256: 43af81a01f25a7771c9a42809a31625f4d0de3f335cdf29961f9ba179a86c13d
Python: 3.12.4
pandas: 2.2.2
numpy: 1.26.4
seaborn: 0.13.2
Sistema: Windows
Dimensiones: (891, 15)


### 3.2 Unidad de análisis, variables y estado general

Cada fila representa un registro de pasajero. La caracterización inicial considera dimensiones, nombres y tipos de variables, valores faltantes, cardinalidad, rangos numéricos, categorías observadas y coincidencias completas entre filas. Estas comprobaciones permiten establecer el estado general del dataset y fundamentar las decisiones metodológicas del EDA.


In [2]:
estructura = pd.DataFrame({
    "variable": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "nulos": df.isna().sum().values,
    "porcentaje_nulos": (df.isna().mean().values * 100).round(2),
    "valores_unicos": [df[c].nunique(dropna=True) for c in df.columns],
})

coincidencias_completas = int(df.duplicated(keep=False).sum())
filas_duplicadas_pandas = int(df.duplicated().sum())

print("Primeros registros:")
display(df.head())

print("Caracterización de variables:")
display(estructura)

print("Filas posteriores coincidentes según df.duplicated():", filas_duplicadas_pandas)
print("Registros involucrados en coincidencias completas:", coincidencias_completas)

estructura.to_csv(OUTPUTS_DIR / "caracterizacion_dataset.csv", index=False, encoding="utf-8-sig")


Primeros registros:


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


Caracterización de variables:


,variable,tipo,nulos,porcentaje_nulos,valores_unicos
0,survived,int64,0,0.00,2
1,pclass,int64,0,0.00,3
2,sex,object,0,0.00,2
3,age,float64,177,19.87,88
4,sibsp,int64,0,0.00,7
5,parch,int64,0,0.00,7
6,fare,float64,0,0.00,248
7,embarked,object,2,0.22,3
8,class,object,0,0.00,3
9,who,object,0,0.00,3


Filas posteriores coincidentes según df.duplicated(): 107
Registros involucrados en coincidencias completas: 160


In [3]:
numericas = df.select_dtypes(include="number")
rangos = pd.DataFrame({
    "variable": numericas.columns,
    "minimo": numericas.min().values,
    "maximo": numericas.max().values,
    "mediana": numericas.median().values,
})

categoricas_objetivo = ["sex", "embarked", "class", "who", "deck", "embark_town", "alive", "alone"]
categorias = []
for var in categoricas_objetivo:
    if var in df.columns:
        categorias.append({
            "variable": var,
            "categorias_observadas": ", ".join(map(str, sorted(df[var].dropna().astype(str).unique())))
        })
categorias = pd.DataFrame(categorias)

print("Rangos descriptivos iniciales:")
display(rangos)
print("Categorías observadas:")
display(categorias)

rangos.to_csv(OUTPUTS_DIR / "rangos_iniciales.csv", index=False, encoding="utf-8-sig")
categorias.to_csv(OUTPUTS_DIR / "categorias_iniciales.csv", index=False, encoding="utf-8-sig")


Rangos descriptivos iniciales:


,variable,minimo,maximo,mediana
0,survived,0.00,1.0000,0.0000
1,pclass,1.00,3.0000,3.0000
2,age,0.42,80.0000,28.0000
3,sibsp,0.00,8.0000,0.0000
4,parch,0.00,6.0000,0.0000
5,fare,0.00,512.3292,14.4542


Categorías observadas:


,variable,categorias_observadas
0,sex,"female, male"
1,embarked,"C, Q, S"
2,class,"First, Second, Third"
3,who,"child, man, woman"
4,deck,"A, B, C, D, E, F, G"
5,embark_town,"Cherbourg, Queenstown, Southampton"
6,alive,"no, yes"
7,alone,"False, True"


In [4]:
faltantes = estructura.loc[estructura["nulos"] > 0, ["variable", "nulos", "porcentaje_nulos"]]

display(Markdown(
    "### Observaciones iniciales del dataset\n\n"
    f"- La copia analizada contiene **{df.shape[0]} registros y {df.shape[1]} variables**.\n"
    f"- Se identifican variables con datos faltantes: "
    + ", ".join(
        f"`{r.variable}` ({int(r.nulos)}; {r.porcentaje_nulos:.2f} %)"
        for r in faltantes.itertuples()
    )
    + ".\n"
    f"- `df.duplicated()` identifica **{filas_duplicadas_pandas} filas posteriores coincidentes**. "
    "Como el dataset de Seaborn no incorpora un identificador individual único, estas coincidencias "
    "se documentan como una condición a revisar y no se eliminan automáticamente.\n"
    "- Las variables disponibles permiten planificar análisis descriptivos de supervivencia por sexo, "
    "clase, edad, tarifa y características familiares.\n"
))


### Observaciones iniciales del dataset

- La copia analizada contiene **891 registros y 15 variables**.
- Se identifican variables con datos faltantes: `age` (177; 19.87 %), `embarked` (2; 0.22 %), `deck` (688; 77.22 %), `embark_town` (2; 0.22 %).
- `df.duplicated()` identifica **107 filas posteriores coincidentes**. Como el dataset de Seaborn no incorpora un identificador individual único, estas coincidencias se documentan como una condición a revisar y no se eliminan automáticamente.
- Las variables disponibles permiten planificar análisis descriptivos de supervivencia por sexo, clase, edad, tarifa y características familiares.


## IV. Preguntas exploratorias iniciales, supuestos y alcance

### Preguntas exploratorias

**P1.** ¿Cuál es la proporción observada de supervivencia y cómo varía descriptivamente según `sex`?  

**P2.** ¿Existen diferencias descriptivas de supervivencia entre las categorías de `pclass` y cómo se comportan al considerar conjuntamente `sex` y `pclass`?  

**P3.** ¿Cómo se distribuye `age` y qué diferencias descriptivas pueden evaluarse entre sobrevivientes y no sobrevivientes?  

**P4.** ¿Cómo se comporta `fare` respecto de `pclass` y `survived`, y qué valores extremos requieren revisión antes de definir un tratamiento?  

**P5.** ¿El tamaño del grupo familiar construido a partir de `sibsp` y `parch` presenta diferencias descriptivas de supervivencia?

### Supuestos iniciales

- `survived = 1` representa supervivencia y `survived = 0` no supervivencia.
- `pclass` se analizará como variable ordinal/categórica.
- Los valores faltantes no serán imputados ni eliminados sin diagnóstico y justificación.
- Un valor extremo estadístico no se considerará automáticamente un error de registro.
- Las asociaciones observadas se interpretarán de manera descriptiva, sin atribución causal.

### Alcance y limitaciones preliminares

El análisis se limita a los registros contenidos en la copia auditada del dataset. La ausencia de un identificador individual único condiciona la interpretación de filas completamente coincidentes. Las decisiones de imputación, exclusión de registros o tratamiento de valores extremos deberán fundamentarse en la evidencia obtenida durante el EDA y quedar documentadas.


## V. Organización prevista del análisis

El análisis se organizará mediante un flujo secuencial y trazable:

1. **Configuración e integridad:** cargar la copia local y verificar su identidad.
2. **Diagnóstico de calidad:** revisar tipos, faltantes, rangos, categorías y coincidencias completas.
3. **Descriptivos iniciales:** caracterizar distribuciones relevantes sin inferencia causal.
4. **Preguntas P1–P5:** ejecutar comparaciones univariadas y bivariadas vinculadas al objetivo del proyecto.
5. **Transformaciones justificadas:** construir y evaluar variables derivadas familiares cuando corresponda.
6. **Síntesis y evidencia:** exportar tablas, figuras y una matriz de trazabilidad entre pregunta, acción, resultado y evidencia.
7. **Registro de decisiones:** documentar las decisiones adoptadas, su justificación y los resultados intermedios asociados.

### Transformaciones previstas

Se proyecta construir:

\[
FamilySize = sibsp + parch + 1
\]

y, a partir de ella:

\[
IsAlone =
\begin{cases}
1, & FamilySize = 1 \\
0, & FamilySize > 1
\end{cases}
\]

Aunque Seaborn incluye la variable `alone`, `IsAlone` se rederivará desde `sibsp` y `parch` para mantener trazabilidad explícita de la transformación.


In [5]:
plan_eda = pd.DataFrame([
    {
        "ID": "P1",
        "pregunta": "Supervivencia general y según sex",
        "variables": "survived, sex",
        "accion_prevista": "Calcular tasas y comparar descriptivamente por sexo",
        "evidencia_prevista": "Tabla y visualización"
    },
    {
        "ID": "P2",
        "pregunta": "Supervivencia según pclass y cruce sex × pclass",
        "variables": "survived, pclass, sex",
        "accion_prevista": "Calcular tasas por clase y cruce de categorías",
        "evidencia_prevista": "Tablas y visualización"
    },
    {
        "ID": "P3",
        "pregunta": "Distribución de age y comparación según survived",
        "variables": "age, survived",
        "accion_prevista": "Cuantificar faltantes, descriptivos y distribución",
        "evidencia_prevista": "Tabla y visualización"
    },
    {
        "ID": "P4",
        "pregunta": "Comportamiento de fare respecto de pclass y survived",
        "variables": "fare, pclass, survived",
        "accion_prevista": "Describir distribución, IQR y comparaciones por grupos",
        "evidencia_prevista": "Tablas y visualizaciones"
    },
    {
        "ID": "P5",
        "pregunta": "Tamaño familiar y supervivencia",
        "variables": "sibsp, parch, survived",
        "accion_prevista": "Construir FamilySize e IsAlone y comparar tasas",
        "evidencia_prevista": "Tablas y visualización"
    },
])

display(plan_eda)
plan_eda.to_csv(OUTPUTS_DIR / "plan_eda.csv", index=False, encoding="utf-8-sig")


,ID,pregunta,variables,accion_prevista,evidencia_prevista
0,P1,Supervivencia general y según sex,"survived, sex",Calcular tasas y comparar descriptivamente por...,Tabla y visualización
1,P2,Supervivencia según pclass y cruce sex × pclass,"survived, pclass, sex",Calcular tasas por clase y cruce de categorías,Tablas y visualización
2,P3,Distribución de age y comparación según survived,"age, survived","Cuantificar faltantes, descriptivos y distribu...",Tabla y visualización
3,P4,Comportamiento de fare respecto de pclass y su...,"fare, pclass, survived","Describir distribución, IQR y comparaciones po...",Tablas y visualizaciones
4,P5,Tamaño familiar y supervivencia,"sibsp, parch, survived",Construir FamilySize e IsAlone y comparar tasas,Tablas y visualización


## VI. Criterios iniciales de trabajo reproducible

El trabajo se organizará con los siguientes criterios:

- **Conservación del insumo:** el dataset original se mantiene sin sobrescritura en `data/input/`.
- **Rutas relativas:** el notebook no depende de directorios personales.
- **Integridad del dato:** el archivo de entrada se identifica mediante SHA-256.
- **Transformaciones por código:** las variables derivadas se construirán programáticamente y no mediante edición manual.
- **Separación de productos:** los archivos derivados se almacenarán fuera del dataset original.
- **Documentación técnica:** las celdas Markdown registrarán objetivo, supuestos, decisiones, limitaciones y secuencia analítica.
- **Entorno documentado:** las versiones relevantes del entorno se registrarán en un manifiesto y en archivos de dependencias.
- **Trazabilidad analítica:** las preguntas P1–P5 se vincularán con acciones, resultados y evidencia verificable.


In [6]:
from importlib.metadata import version, PackageNotFoundError

def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "no disponible"

manifest = pd.DataFrame([
    ("dataset", "data/input/titanic_seaborn.csv"),
    ("sha256", sha256),
    ("filas", df.shape[0]),
    ("columnas", df.shape[1]),
    ("python", sys.version.split()[0]),
    ("pandas", pd.__version__),
    ("numpy", np.__version__),
    ("seaborn", sns.__version__),
    ("ipykernel", package_version("ipykernel")),
    ("notebook", package_version("notebook")),
    ("jupyterlab", package_version("jupyterlab")),
], columns=["elemento", "valor"])

manifest.to_csv(OUTPUTS_DIR / "manifest_formativa.csv", index=False, encoding="utf-8-sig")
display(manifest)


,elemento,valor
0,dataset,data/input/titanic_seaborn.csv
1,sha256,43af81a01f25a7771c9a42809a31625f4d0de3f335cdf2...
2,filas,891
3,columnas,15
4,python,3.12.4
5,pandas,2.2.2
6,numpy,1.26.4
7,seaborn,0.13.2
8,ipykernel,6.28.0
9,notebook,7.0.8


In [7]:
resumen_path = OUTPUTS_DIR / "resumen_para_ficha.md"
faltantes_txt = ", ".join(
    f"{r.variable}: {int(r.nulos)} ({r.porcentaje_nulos:.2f} %)"
    for r in faltantes.itertuples()
)

resumen = f"""# Resumen técnico para la ficha formativa

- Dataset: Seaborn `titanic`.
- Unidad de análisis: registro de pasajero.
- Dimensiones: {df.shape[0]} registros × {df.shape[1]} variables.
- SHA-256: `{sha256}`.
- Variables con faltantes: {faltantes_txt}.
- Filas posteriores coincidentes según `df.duplicated()`: {filas_duplicadas_pandas}.
- Alcance: caracterización inicial del dataset y planificación del EDA reproducible.
"""
resumen_path.write_text(resumen, encoding="utf-8")
print(resumen)


# Resumen técnico para la ficha formativa

- Dataset: Seaborn `titanic`.
- Unidad de análisis: registro de pasajero.
- Dimensiones: 891 registros × 15 variables.
- SHA-256: `43af81a01f25a7771c9a42809a31625f4d0de3f335cdf29961f9ba179a86c13d`.
- Variables con faltantes: age: 177 (19.87 %), embarked: 2 (0.22 %), deck: 688 (77.22 %), embark_town: 2 (0.22 %).
- Filas posteriores coincidentes según `df.duplicated()`: 107.
- Alcance: caracterización inicial del dataset y planificación del EDA reproducible.



In [8]:
checks = {
    "integridad_hash": sha256 == EXPECTED_SHA256,
    "dimensiones_dataset": df.shape == (891, 15),
    "columnas_esperadas": list(df.columns) == [
        "survived", "pclass", "sex", "age", "sibsp", "parch", "fare",
        "embarked", "class", "who", "adult_male", "deck", "embark_town",
        "alive", "alone"
    ],
    "caracterizacion_exportada": (OUTPUTS_DIR / "caracterizacion_dataset.csv").exists(),
    "rangos_exportados": (OUTPUTS_DIR / "rangos_iniciales.csv").exists(),
    "categorias_exportadas": (OUTPUTS_DIR / "categorias_iniciales.csv").exists(),
    "plan_exportado": (OUTPUTS_DIR / "plan_eda.csv").exists(),
    "manifest_exportado": (OUTPUTS_DIR / "manifest_formativa.csv").exists(),
    "resumen_exportado": resumen_path.exists(),
}

validacion = pd.DataFrame({
    "control": checks.keys(),
    "estado": ["OK" if v else "ERROR" for v in checks.values()]
})
display(validacion)

estado_global = all(checks.values())
(OUTPUTS_DIR / "validacion_formativa.txt").write_text(
    "\n".join(f"{k}={'OK' if v else 'ERROR'}" for k, v in checks.items())
    + f"\nestado_global={'OK' if estado_global else 'ERROR'}\n",
    encoding="utf-8"
)

if not estado_global:
    raise AssertionError("La validación técnica detectó inconsistencias.")

print("VALIDACIÓN FORMATIVA: OK")


,control,estado
0,integridad_hash,OK
1,dimensiones_dataset,OK
2,columnas_esperadas,OK
3,caracterizacion_exportada,OK
4,rangos_exportados,OK
5,categorias_exportadas,OK
6,plan_exportado,OK
7,manifest_exportado,OK
8,resumen_exportado,OK


VALIDACIÓN FORMATIVA: OK


## VII. Bibliografía

Bruce, P., Bruce, A., & Gedeck, P. (2020). *Practical statistics for data scientists: 50+ essential concepts using R and Python* (2nd ed.). O’Reilly Media.

Paraíso, S. (2026). *Checklist notebook reproducible* [Infografía]. Universidad Andrés Bello.

The pandas development team. (2024). *pandas documentation*. https://pandas.pydata.org/

Waskom, M. L. (2021). seaborn: Statistical data visualization. *Journal of Open Source Software, 6*(60), 3021. https://doi.org/10.21105/joss.03021
